# K-Means Clustering on an Unlabelled Dataset



**Dataset:** https://www.kaggle.com/datasets/ahmedaliraja/unlabelled-dataset/data



## 1. Definitions

### What is K-Means?
**K-Means** is an unsupervised clustering algorithm that divides data into **k groups (clusters)**. It repeatedly:
1. Assigns every data point to the nearest cluster centroid.
2. Recalculates each centroid as the mean of the points assigned to it.
3. Repeats these steps until the cluster assignments stabilize or the maximum number of iterations is reached.

K-Means requires the value of **k** to be chosen beforehand.

### What is an Elbow Curve?
The **Elbow Curve** plots **inertia (within-cluster sum of squares)** against different values of k.

Inertia measures how close the data points are to their assigned cluster centroids. It always tends to decrease as k increases. The useful value of k is often near the point where the decrease begins to slow noticeably — the **elbow**.

### What is Silhouette Score?
The **Silhouette Score** measures how well-separated and compact the clusters are.

It ranges from **-1 to +1**:
- Close to **+1** → points are well matched to their own cluster and far from other clusters.
- Around **0** → clusters overlap.
- Below **0** → points may have been assigned to the wrong cluster.

A higher silhouette score is generally better.

### What is Principal Component Analysis (PCA)?
**PCA** is a dimensionality-reduction technique. It transforms the original features into new **principal components** that capture as much of the variation in the data as possible.

For this experiment, PCA reduces the dataset to **2 components** so that the clusters can be visualized on a 2D scatter plot.


## 2. Import Required Libraries

In [ ]:
import os
import glob
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 3. Download / Load the Kaggle Dataset

The notebook first tries the Kaggle API download endpoint. If that is unavailable, use the upload cell to upload the CSV downloaded from Kaggle.

Dataset:
https://www.kaggle.com/datasets/ahmedaliraja/unlabelled-dataset/data


In [ ]:
# Automatic Kaggle download
# Dataset slug: ahmedaliraja/unlabelled-dataset

dataset_dir = "/content/unlabelled_dataset"
os.makedirs(dataset_dir, exist_ok=True)

csv_candidates = []

try:
    import requests

    url = "https://www.kaggle.com/api/v1/datasets/download/ahmedaliraja/unlabelled-dataset"
    zip_path = "/content/unlabelled_dataset.zip"

    response = requests.get(url, timeout=30)
    print("Kaggle response status:", response.status_code)

    if response.ok:
        with open(zip_path, "wb") as f:
            f.write(response.content)

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(dataset_dir)

        csv_candidates = glob.glob(dataset_dir + "/**/*.csv", recursive=True)
        print("CSV files found:", csv_candidates)
    else:
        print("Automatic download was not available.")
except Exception as e:
    print("Automatic download failed:", e)

if not csv_candidates:
    print("Please run the upload cell below.")


In [ ]:
# Fallback: manually upload the CSV from Kaggle

from google.colab import files

if not csv_candidates:
    uploaded = files.upload()
    csv_candidates = [name for name in uploaded.keys()
                      if name.lower().endswith(".csv")]

if not csv_candidates:
    raise FileNotFoundError(
        "No CSV file found. Please download the dataset from Kaggle and upload its CSV file."
    )

file_path = csv_candidates[0]
print("Using:", file_path)

df = pd.read_csv(file_path)
print("Dataset shape:", df.shape)
display(df.head())


## 4. Inspect the Dataset

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nSummary statistics:")
display(df.describe(include="all").T)


## 5. Prepare Features for K-Means

K-Means uses numerical distances, so categorical/text columns cannot be passed directly to the algorithm.

The following preprocessing is performed automatically:
- Keep numerical columns.
- Remove obvious ID/index columns where appropriate.
- Remove columns with only one unique value.
- Fill missing numerical values using the median.
- Standardize all features using `StandardScaler`.

**Why standardize?**  
K-Means is distance-based. Without scaling, a feature with a much larger numerical range can dominate the distance calculation.


In [ ]:
# Automatically select useful numerical features

numeric_df = df.select_dtypes(include=np.number).copy()

# Drop columns that look like identifiers/indexes.
id_like_cols = []
for col in numeric_df.columns:
    col_lower = str(col).lower()
    unique_ratio = numeric_df[col].nunique(dropna=True) / max(len(numeric_df), 1)

    if (
        col_lower in {"id", "index"} or
        "customerid" in col_lower or
        col_lower.endswith("_id") or
        (unique_ratio > 0.98 and numeric_df[col].nunique(dropna=True) > 20)
    ):
        id_like_cols.append(col)

numeric_df = numeric_df.drop(columns=id_like_cols, errors="ignore")

# Remove constant columns
constant_cols = [
    col for col in numeric_df.columns
    if numeric_df[col].nunique(dropna=True) <= 1
]
numeric_df = numeric_df.drop(columns=constant_cols, errors="ignore")

# Fill missing values
numeric_df = numeric_df.fillna(numeric_df.median(numeric_only=True))

print("Dropped ID-like columns:", id_like_cols if id_like_cols else "None")
print("Dropped constant columns:", constant_cols if constant_cols else "None")
print("Features used for clustering:", numeric_df.columns.tolist())

if numeric_df.shape[1] < 2:
    raise ValueError(
        "At least two useful numerical features are required for this experiment."
    )

scaler = StandardScaler()
X_scaled = scaler.fit_transform(numeric_df)

print("\nFinal feature matrix shape:", X_scaled.shape)


## 6. Run K-Means for Different Values of k

We test values from **k = 2 to k = 10**.

For every k:
- Fit K-Means.
- Record **inertia** for the Elbow Curve.
- Calculate the **Silhouette Score**.


In [ ]:
k_range = range(2, 11)

inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_scaled)

    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

results = pd.DataFrame({
    "k": list(k_range),
    "Inertia": inertias,
    "Silhouette Score": silhouette_scores
})

display(results)


# 7. Elbow Curve — Inertia vs k

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(results["k"], results["Inertia"], marker="o")
plt.title("Elbow Curve: Inertia vs k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia (Within-Cluster Sum of Squares)")
plt.xticks(list(k_range))
plt.show()


### Elbow Analysis

As k increases, inertia decreases because each point has more potential centroids available.

The **elbow** is the point where increasing k further produces relatively smaller improvements. This provides a candidate value of k.

However, inertia alone is not enough because it always decreases as more clusters are added. Therefore, the silhouette plot below is also considered.


# 8. Silhouette Score vs k

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(results["k"], results["Silhouette Score"], marker="o")
best_silhouette_k = int(results.loc[results["Silhouette Score"].idxmax(), "k"])
best_silhouette = float(results["Silhouette Score"].max())

plt.axvline(
    best_silhouette_k,
    linestyle="--",
    label=f"Highest silhouette: k={best_silhouette_k}"
)

plt.scatter(
    [best_silhouette_k],
    [best_silhouette],
    s=100,
    zorder=5
)

plt.title("Silhouette Score vs k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.xticks(list(k_range))
plt.legend()
plt.show()

print(f"Highest silhouette score occurs at k={best_silhouette_k}")
print(f"Silhouette score = {best_silhouette:.4f}")


# 9. Select the Final k

### Selection rule used in this notebook

The final k is chosen by considering **both**:
1. The approximate elbow in the inertia curve.
2. The k with a strong/high silhouette score.

For reproducibility, the notebook uses the **highest silhouette score as the final tie-breaker**, while the Elbow Curve is used as supporting evidence.

This is preferable to selecting k only because inertia is lowest, since inertia naturally falls as k increases.


In [ ]:
# Automatically select the k with the highest silhouette score.
# The Elbow plot is retained as supporting evidence.

final_k = int(results.loc[results["Silhouette Score"].idxmax(), "k"])

print("Selected final k:", final_k)
print(
    f"Reason: k={final_k} has the highest silhouette score "
    f"({results.loc[results['k']==final_k, 'Silhouette Score'].iloc[0]:.4f}) "
    "among the tested values."
)

display(results.sort_values("Silhouette Score", ascending=False).reset_index(drop=True))


## 10. Final K-Means Model

In [ ]:
final_kmeans = KMeans(
    n_clusters=final_k,
    random_state=42,
    n_init=10
)

cluster_labels = final_kmeans.fit_predict(X_scaled)

df_clustered = df.copy()
df_clustered["Cluster"] = cluster_labels

print("Final model:")
print(final_kmeans)
print("\nCluster sizes:")
display(df_clustered["Cluster"].value_counts().sort_index().to_frame("Number of Samples"))


## 11. Cluster Profile

This table shows the average numerical feature values for each discovered cluster.

Because the dataset is unlabelled, these cluster numbers are **not class labels**. For example, Cluster 0 does not inherently mean "best" or "worst"; the meaning has to be interpreted from its feature values.


In [ ]:
cluster_profile = df_clustered.groupby("Cluster")[numeric_df.columns].mean()
display(cluster_profile)


# 12. Principal Component Analysis (PCA)

PCA reduces the standardized feature matrix to **two principal components** for visualization.

The clusters are still created using the original standardized feature space. PCA is being used here primarily to make the high-dimensional clustering result visible in 2D.


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained_variance = pca.explained_variance_ratio_

print("Explained variance by PC1:", f"{explained_variance[0]*100:.2f}%")
print("Explained variance by PC2:", f"{explained_variance[1]*100:.2f}%")
print(
    "Total variance retained by the two components:",
    f"{explained_variance.sum()*100:.2f}%"
)


# 13. 2D PCA Scatter Plot Colored by Cluster

In [ ]:
plt.figure(figsize=(10, 7))

for cluster in sorted(np.unique(cluster_labels)):
    mask = cluster_labels == cluster

    plt.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        s=55,
        alpha=0.8,
        label=f"Cluster {cluster}"
    )

# Transform the K-Means centroids into PCA space
centroids_pca = pca.transform(final_kmeans.cluster_centers_)

plt.scatter(
    centroids_pca[:, 0],
    centroids_pca[:, 1],
    marker="X",
    s=220,
    edgecolor="black",
    linewidth=1.5,
    label="Centroids"
)

plt.title(f"K-Means Clusters Visualized with 2D PCA (k={final_k})")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


# 14. Final Results Summary

In [ ]:
final_row = results[results["k"] == final_k].iloc[0]

print("=" * 55)
print("FINAL K-MEANS CLUSTERING RESULTS")
print("=" * 55)
print(f"Number of samples        : {len(df)}")
print(f"Features used            : {numeric_df.shape[1]}")
print(f"Selected k               : {final_k}")
print(f"Final inertia            : {final_row['Inertia']:.4f}")
print(f"Final silhouette score   : {final_row['Silhouette Score']:.4f}")
print(f"PCA variance retained    : {explained_variance.sum()*100:.2f}%")
print("=" * 55)

print("\nCluster sizes:")
display(df_clustered["Cluster"].value_counts().sort_index().to_frame("Samples"))


# 15. Conclusion

### Key findings

- **K-Means** groups unlabelled observations according to similarity/distance from cluster centroids.
- The **Elbow Curve** helps identify where adding more clusters begins to provide diminishing returns.
- The **Silhouette Score** evaluates cluster compactness and separation; higher values are generally preferable.
- Both measures are considered when selecting the final **k**.
- **PCA** reduces the feature space to two dimensions so the resulting clusters can be visualized.
- The final PCA scatter plot shows how the discovered clusters are distributed in the reduced 2D space.

### Important note

Because this is an **unlabelled dataset**, there is no ground-truth target variable for ordinary classification accuracy. Therefore, clustering quality is evaluated using **inertia and silhouette score**, along with visual inspection of the PCA plot.


## Deliverables Completed

| Requirement | Notebook Output |
|---|---|
| Define K-Means | ✅ |
| Define Elbow Curve | ✅ |
| Define Silhouette | ✅ |
| Define PCA | ✅ |
| Load unlabelled Kaggle dataset | ✅ |
| K-Means for different k values | ✅ |
| Elbow curve | ✅ |
| Silhouette score plot | ✅ |
| Final k selection | ✅ |
| 2D PCA cluster scatter plot | ✅ |
| Cluster analysis | ✅ |
